# Two Towers From Scratch for Learning Equality

**Course:** Session 16 of [Master Transformers from Scratch: BERT, ViT & AI Search](https://www.udemy.com/course/master-transformers-from-scratch-bert-vit-ai-search/)
**Competition:** [Learning Equality — Curriculum Recommendations](https://www.kaggle.com/competitions/learning-equality-curriculum-recommendations)

We build a semantic search system for curriculum content. Given a curriculum **topic**,
return the **content** items that belong with it.

This notebook deliberately avoids `sentence-transformers`. We build both towers ourselves
from a plain transformer encoder so every step is visible:

| Cell | What it does |
|------|--------------|
| 1 | Setup and constants |
| 2 | Load the competition CSVs |
| 3 | Build one text string per topic and per content item |
| 4 | Build labeled pairs — the `(topic, content, label)` table |
| 5 | Tokenizer, Dataset, collate |
| 6 | The `TwoTower` model — two encoders, pooling, L2 normalize |
| 7 | Three losses — contrastive, triplet, in-batch cross-entropy |
| 8 | Train the towers |
| 9 | Encode the content library (the offline index) |
| 10 | Retrieve top-k per topic, measure recall@k |
| 11 | Cross-encoder — one encoder over `[CLS] topic [SEP] content [SEP]` |
| 12 | Re-rank the shortlist |
| 13 | Write `submission.csv` |

**Turn on a GPU and turn on Internet** so the pretrained weights can download.

## Cell 1 — Setup and constants

Everything tunable lives here. `DEMO_MODE` keeps the run inside a classroom time budget by
using a subset of the content library; set it to `False` for a fuller run.

In [ ]:
import os, glob, math, random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModel

# Every from_pretrained otherwise prints a long "LOAD REPORT" table of unexpected
# keys, which drowns the real output in the cell screenshots.
transformers.logging.set_verbosity_error()

SEED = 42
MODEL_NAME = "distilbert-base-multilingual-cased"

MAX_LEN = 128          # tokens per side
BATCH_SIZE = 16
LR = 2e-5
NUM_EPOCHS = 1
TEMPERATURE = 0.05     # scales similarities before the in-batch softmax
NEG_PER_POS = 1        # sampled negatives per positive, for the labeled table

TOP_K_RETRIEVE = 50    # candidates the towers hand to the re-ranker
TOP_K_SUBMIT = 5       # content ids we finally write per topic

CROSS_STEPS = 2000     # re-ranker training steps
CROSS_POS_PER_TOPIC = 4
CROSS_NEG_PER_TOPIC = 8

DEMO_MODE = True
DEMO_CONTENT_ROWS = 8000
DEMO_TRAIN_PAIRS = 4000
DEMO_PRED_TOPICS = 150      # eval topics; more topics = a less noisy precision
DEMO_RERANK_TOPICS = 2500   # topics used ONLY to train the re-ranker

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device    =", DEVICE)
print("model     =", MODEL_NAME)
print("DEMO_MODE =", DEMO_MODE)

## Cell 2 — Load the competition CSVs

On Kaggle the data mounts under `/kaggle/input`. The folder name can vary, so we glob for
`topics.csv` instead of hardcoding a path.

`content.csv` is large (about 870 MB), so we read only the columns we actually need.

In [ ]:
print("contents of /kaggle/input:",
      os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "not on Kaggle")

candidates = sorted(glob.glob("/kaggle/input/**/topics.csv", recursive=True)) + ["./topics.csv"]
DATA = next(os.path.dirname(c) or "." for c in candidates if os.path.exists(c))
print("using DATA =", DATA)

topics = pd.read_csv(os.path.join(DATA, "topics.csv"))
corr = pd.read_csv(os.path.join(DATA, "correlations.csv"))
sample_sub = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))

content = pd.read_csv(
    os.path.join(DATA, "content.csv"),
    usecols=["id", "title", "description", "text", "language", "kind"],
    dtype={"id": "string", "language": "string", "kind": "string"},
)

print("topics      :", topics.shape)
print("content     :", content.shape)
print("correlations:", corr.shape)
print()
print("one correlation row (topic_id -> content_ids):")
print(corr.head(2).to_string(index=False))

## Cell 3 — One text string per topic and per content item

Each tower consumes plain text, so we flatten the useful columns into a single string.
Title plus description is a strong, simple recipe. Rows whose text is empty cannot produce
a meaningful embedding, so we drop them.

In [ ]:
def join_text(*parts):
    out = []
    for p in parts:
        if pd.isna(p):
            continue
        s = str(p).strip()
        if s:
            out.append(s)
    return " ".join(out)

topics["topic_text"] = [join_text(t, d) for t, d in zip(topics["title"], topics["description"])]
content["content_text"] = [join_text(t, d, x) for t, d, x in
                           zip(content["title"], content["description"], content["text"])]

topics = topics[topics["topic_text"].str.len() > 0].reset_index(drop=True)
content = content[content["content_text"].str.len() > 0].reset_index(drop=True)

print("topics with text :", len(topics))
print("content with text:", len(content))
print()
print("example topic  :", topics["topic_text"].iloc[0][:120])
print("example content:", content["content_text"].iloc[0][:120])

## Cell 4 — Labeled pairs: the training signal

This is the table the whole training phase rests on. Every link in `correlations.csv`
becomes a **positive** pair with label `1`. For each positive we sample a content item that
is *not* linked to that topic and label it `0`.

The label is the only thing that tells the model which way to move the two embeddings.

In [ ]:
topic_text_by_id = dict(zip(topics["id"], topics["topic_text"]))
content_text_by_id = dict(zip(content["id"], content["content_text"]))
content_lang_by_id = dict(zip(content["id"], content["language"]))
all_content_ids = content["id"].tolist()

rows = []
for tid, cids in zip(corr["topic_id"], corr["content_ids"]):
    if tid not in topic_text_by_id:
        continue
    positives = [c for c in str(cids).split() if c in content_text_by_id]
    if not positives:
        continue
    positive_set = set(positives)
    for cid in positives:
        rows.append((tid, cid, 1))
        for _ in range(NEG_PER_POS):
            neg = random.choice(all_content_ids)
            while neg in positive_set:
                neg = random.choice(all_content_ids)
            rows.append((tid, neg, 0))

pairs = pd.DataFrame(rows, columns=["topic_id", "content_id", "label"])
pairs = pairs.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

if DEMO_MODE:
    pairs = pairs.head(DEMO_TRAIN_PAIRS).reset_index(drop=True)

print("labeled pairs:", len(pairs))
print(pairs["label"].value_counts().to_string())
print()

preview = pairs.head(6).copy()
preview["topic"] = [topic_text_by_id[t][:34] for t in preview["topic_id"]]
preview["content"] = [content_text_by_id[c][:34] for c in preview["content_id"]]
print(preview[["topic", "content", "label"]].to_string(index=False))

## Cell 5 — Tokenizer, Dataset, collate

Each tower tokenizes its own side independently. Nothing is concatenated here — that
separation is exactly what lets us precompute content embeddings later.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDataset(Dataset):
    """Yields (topic_text, content_text, label) triples."""

    def __init__(self, frame):
        self.topic = [topic_text_by_id[t] for t in frame["topic_id"]]
        self.content = [content_text_by_id[c] for c in frame["content_id"]]
        self.label = frame["label"].to_numpy(dtype="float32")

    def __len__(self):
        return len(self.label)

    def __getitem__(self, i):
        return self.topic[i], self.content[i], self.label[i]


def collate(batch):
    q_texts, d_texts, labels = zip(*batch)
    enc = lambda texts: tokenizer(list(texts), padding=True, truncation=True,
                                  max_length=MAX_LEN, return_tensors="pt")
    return enc(q_texts), enc(d_texts), torch.tensor(labels)


positives_only = pairs[pairs["label"] == 1].reset_index(drop=True)
train_loader = DataLoader(PairDataset(positives_only), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate, drop_last=True)
labeled_loader = DataLoader(PairDataset(pairs), batch_size=BATCH_SIZE,
                            shuffle=True, collate_fn=collate, drop_last=True)

q_batch, d_batch, y_batch = next(iter(labeled_loader))
print("query input_ids   :", tuple(q_batch["input_ids"].shape))
print("content input_ids :", tuple(d_batch["input_ids"].shape))
print("labels            :", y_batch.tolist())

## Cell 6 — The two-tower model

Two separate encoders, both initialized from the *same* pretrained checkpoint. At step zero
they are identical copies; training updates them independently so they drift apart. Same
starting point, different destinations.

After the encoder we still have one vector per token, but search needs one vector per text,
so we **mean pool** over the real (non-padding) tokens and then **L2 normalize**. Once both
vectors are unit length, a dot product *is* cosine similarity.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    """Average token vectors, ignoring padding."""
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class TwoTower(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.query_encoder = AutoModel.from_pretrained(model_name)
        self.doc_encoder = AutoModel.from_pretrained(model_name)

    def encode_query(self, enc):
        out = self.query_encoder(**enc).last_hidden_state
        return F.normalize(mean_pool(out, enc["attention_mask"]), dim=-1)

    def encode_doc(self, enc):
        out = self.doc_encoder(**enc).last_hidden_state
        return F.normalize(mean_pool(out, enc["attention_mask"]), dim=-1)

    def forward(self, q_enc, d_enc):
        return self.encode_query(q_enc), self.encode_doc(d_enc)


model = TwoTower(MODEL_NAME).to(DEVICE)

q_w = model.query_encoder.embeddings.word_embeddings.weight
d_w = model.doc_encoder.embeddings.word_embeddings.weight
print("towers identical before training:", torch.equal(q_w, d_w))

params = sum(p.numel() for p in model.parameters())
print(f"total parameters: {params/1e6:.1f}M  (two encoders)")

with torch.no_grad():
    q, d = model({k: v.to(DEVICE) for k, v in q_batch.items()},
                 {k: v.to(DEVICE) for k, v in d_batch.items()})
print("query embeddings  :", tuple(q.shape))
print("content embeddings:", tuple(d.shape))
print("norm of first query vector:", float(q[0].norm()))

## Cell 7 — Three ways to turn labels into a loss

All three push matching pairs together and non-matching pairs apart. They differ in *what
they compare against*.

1. **Contrastive loss** — one pair at a time, uses the `0/1` label directly. Positives are
   pulled to similarity 1; negatives are pushed below a margin.
2. **Triplet loss** — a topic with one positive *and* one negative at once. It only asks
   that the positive beat the negative by a margin.
3. **In-batch cross-entropy** — the whole batch at once. For each topic, its own content is
   the correct answer and every *other* content vector in the batch is a free negative.
   This is what we train with, because one batch of 16 gives 16 negatives per topic at no
   extra cost.

In [ ]:
def contrastive_loss(q, d, label, margin=0.3):
    """Pairwise. Positives -> similarity 1. Negatives -> similarity below margin."""
    sim = (q * d).sum(dim=-1)
    pos_term = label * (1.0 - sim).pow(2)
    neg_term = (1.0 - label) * F.relu(sim - margin).pow(2)
    return (pos_term + neg_term).mean()


def triplet_loss(q, d_pos, d_neg, margin=0.2):
    """The positive must beat the negative by at least `margin`."""
    sim_pos = (q * d_pos).sum(dim=-1)
    sim_neg = (q * d_neg).sum(dim=-1)
    return F.relu(margin - sim_pos + sim_neg).mean()


def in_batch_cross_entropy(q, d, temperature=TEMPERATURE):
    """Every other content vector in the batch acts as a negative."""
    logits = (q @ d.t()) / temperature          # (B, B) similarity matrix
    targets = torch.arange(q.size(0), device=q.device)   # correct answer is the diagonal
    return F.cross_entropy(logits, targets)


# --- see all three on real batches ---------------------------------------------------
# Contrastive needs the 0/1 labels, so it gets the mixed batch. The other two assume the
# pairing is correct, so they get a positives-only batch.
pos_q, pos_d, _ = next(iter(train_loader))

with torch.no_grad():
    to_dev = lambda enc: {k: v.to(DEVICE) for k, v in enc.items()}
    q_lab, d_lab = model(to_dev(q_batch), to_dev(d_batch))
    q_pos, d_pos = model(to_dev(pos_q), to_dev(pos_d))
    y = y_batch.to(DEVICE)

    print("contrastive (mixed labels)   :", round(float(contrastive_loss(q_lab, d_lab, y)), 4))
    print("triplet     (pos vs shifted) :", round(float(triplet_loss(q_pos, d_pos, d_pos.roll(1, 0))), 4))
    print("in-batch CE (positives only) :", round(float(in_batch_cross_entropy(q_pos, d_pos)), 4))
    print()
    sim = q_pos @ d_pos.t()
    print("similarity matrix, first 5x5 — the diagonal should win after training:")
    print(np.round(sim[:5, :5].cpu().numpy(), 3))

## Cell 8 — Train the towers

One pass with in-batch cross-entropy. Watch the loss fall and the diagonal of the
similarity matrix grow: that is the towers learning to place a topic near its own content.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

model.train()
step, running = 0, []
for epoch in range(NUM_EPOCHS):
    for q_enc, d_enc, _ in train_loader:
        q_enc = {k: v.to(DEVICE) for k, v in q_enc.items()}
        d_enc = {k: v.to(DEVICE) for k, v in d_enc.items()}

        optimizer.zero_grad(set_to_none=True)
        q, d = model(q_enc, d_enc)
        loss = in_batch_cross_entropy(q, d)
        loss.backward()
        optimizer.step()

        running.append(loss.detach().item())
        step += 1
        if step % 20 == 0:
            print(f"step {step:4d}  loss {np.mean(running[-20:]):.4f}")

print("\ntraining done. steps =", step)
print(f"first 20 steps avg {np.mean(running[:20]):.4f} -> last 20 steps avg {np.mean(running[-20:]):.4f}")

q_w = model.query_encoder.embeddings.word_embeddings.weight
d_w = model.doc_encoder.embeddings.word_embeddings.weight
print("towers identical after training:", torch.equal(q_w, d_w))

## Cell 9 — Encode the content library (the offline index)

This is the two-tower payoff. A content embedding does not depend on any query, so we
encode the library **once** and reuse it for every future topic. In production these
vectors go into a vector store such as FAISS, Qdrant, Pinecone, Weaviate, or pgvector.

We also pick the topics we will work with here, and we split them into two **disjoint**
groups. `eval_topics` is what we measure on. `rerank_topics` exists only to train the
re-ranker in Cell 11, which learns from the retriever's own shortlists — if those were the
same topics we score on, we would be grading the model on its own training data.

In demo mode the index is a subset. It deliberately includes the true content for both
groups plus thousands of distractors, because otherwise the correct answer would not be in
the index at all and recall would be meaningless. In full mode the index is the whole
library.

In [ ]:
gold = {t: set(str(c).split()) for t, c in zip(corr["topic_id"], corr["content_ids"])}
labeled_topics = topics[topics["id"].isin(gold)].reset_index(drop=True)

n_eval = DEMO_PRED_TOPICS if DEMO_MODE else 2000
n_rerank = DEMO_RERANK_TOPICS if DEMO_MODE else 8000
eval_topics = labeled_topics.iloc[:n_eval].reset_index(drop=True)
rerank_topics = labeled_topics.iloc[n_eval:n_eval + n_rerank].reset_index(drop=True)

if DEMO_MODE:
    answers = set()
    for tid in pd.concat([eval_topics["id"], rerank_topics["id"]]):
        answers |= gold[tid]
    is_answer = content["id"].isin(answers)
    distractors = content[~is_answer].sample(
        n=min(DEMO_CONTENT_ROWS, int((~is_answer).sum())), random_state=SEED)
    index_content = (pd.concat([content[is_answer], distractors])
                     .drop_duplicates("id").reset_index(drop=True))
else:
    index_content = content

print("eval topics    :", len(eval_topics))
print("re-rank topics :", len(rerank_topics), "(disjoint from eval)")
print("index rows     :", len(index_content))


@torch.no_grad()
def encode_texts(texts, encode_fn, batch_size=64):
    model.eval()
    vectors = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        vectors.append(encode_fn(enc).float().cpu())
    return torch.cat(vectors)


content_emb = encode_texts(index_content["content_text"].tolist(), model.encode_doc)
content_ids = np.array(index_content["id"].tolist())
content_langs = np.array(index_content["language"].tolist())

print("content index :", tuple(content_emb.shape))
print("index memory  :", round(content_emb.numel() * 4 / 1e6, 1), "MB")

## Cell 10 — Retrieve top-k, then measure recall

A query is embedded **online**, then compared against the whole stored index. Because both
sides are L2-normalized, a single matrix multiply gives cosine similarity against every
content item at once.

This brute-force scan is what approximate nearest neighbour libraries replace at scale.
Recall@k is the right metric here: stage one only has to *include* the right answers.

In [ ]:
def retrieve(topic_vector, language, k=TOP_K_RETRIEVE):
    """Cosine similarity against the whole index, restricted to the same language."""
    allowed = np.where(content_langs == language)[0]
    if len(allowed) == 0:
        allowed = np.arange(len(content_ids))
    sims = content_emb[allowed] @ topic_vector
    top = torch.topk(sims, k=min(k, len(allowed)))
    picked = top.indices.numpy()
    return content_ids[allowed[picked]].tolist(), top.values.numpy()


def retrieve_all(frame):
    emb = encode_texts(frame["topic_text"].tolist(), model.encode_query)
    ids, sims = {}, {}
    for vec, (_, row) in zip(emb, frame.iterrows()):
        ids[row["id"]], sims[row["id"]] = retrieve(vec, row["language"])
    return ids, sims


retrieved, retrieved_sims = retrieve_all(eval_topics)   # measured and submitted
rerank_pool, _ = retrieve_all(rerank_topics)            # feeds Cell 11
print("shortlists built:", len(retrieved), "eval +", len(rerank_pool), "re-rank")


def recall_at_k(mapping):
    hits = total = 0
    for tid, got in mapping.items():
        truth = gold.get(tid, set())
        hits += len(truth & set(got))
        total += len(truth)
    return hits / max(total, 1), hits, total


rate, hits, total = recall_at_k(retrieved)
print(f"recall@{TOP_K_RETRIEVE} = {rate:.3f}   ({hits}/{total} true links found)")

example = eval_topics.iloc[0]
print("\ntopic:", example["topic_text"][:90])
for cid in retrieved[example["id"]][:5]:
    mark = "OK " if cid in gold.get(example["id"], set()) else "   "
    print(f"  {mark}{content_text_by_id.get(cid, '')[:80]}")

## Cell 11 — Cross-encoder: one encoder, both texts together

Same transformer, different wiring. Instead of two separate passes we concatenate the two
texts into **one** sequence separated by `[SEP]`, so tokens from the topic can attend to
tokens from the content. We read the `[CLS]` vector and push it through a linear head to
get one relevance score.

Note that DistilBERT has no segment embeddings, so the `[SEP]` token alone marks the
boundary. Full BERT would additionally add a segment id to every token.

**The negatives matter more than the architecture here.** A re-ranker only ever sees
candidates the towers already liked, so training it on *random* content teaches it nothing
useful — rejecting unrelated material is easy, and it never learns to separate the
plausible-but-wrong from the correct. So we mine **hard negatives**: items that the
retriever actually returned for a topic but that `correlations.csv` does not list. Those
come from `rerank_pool`, whose topics never appear in the evaluation set.

In [ ]:
class CrossEncoder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, enc):
        out = self.encoder(**enc).last_hidden_state
        cls_vector = out[:, 0]                 # [CLS] sits at position 0
        return self.classifier(cls_vector).squeeze(-1)


def collate_pairs(batch):
    q_texts, d_texts, labels = zip(*batch)
    enc = tokenizer(list(q_texts), list(d_texts), padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return enc, torch.tensor(labels)


# --- mine training pairs from the retriever's own shortlists -------------------------
# Both labels must be drawn from the SAME pool, namely the candidates the retriever
# returned. If positives came from the answer key at large while negatives came from the
# shortlist, the model would simply learn "looks retrieved -> wrong", which is inverted at
# inference time when every candidate is retrieved.
hard_rows = []
skipped = 0
for tid, candidates in rerank_pool.items():
    truth = gold.get(tid, set())
    positives = [c for c in candidates if c in truth][:CROSS_POS_PER_TOPIC]
    # Wrong, but the retriever ranked them highly. That is what makes them hard.
    negatives = [c for c in candidates if c not in truth][:CROSS_NEG_PER_TOPIC]
    if not positives or not negatives:
        skipped += 1
        continue
    hard_rows += [(tid, c, 1) for c in positives]
    hard_rows += [(tid, c, 0) for c in negatives]

print("topics skipped (no retrieved positive):", skipped)

cross_pairs = (pd.DataFrame(hard_rows, columns=["topic_id", "content_id", "label"])
               .sample(frac=1.0, random_state=SEED).reset_index(drop=True))

print("re-ranker training pairs:", len(cross_pairs))
print(cross_pairs["label"].value_counts().to_string())

cross_loader = DataLoader(PairDataset(cross_pairs), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_pairs, drop_last=True)

demo_enc, demo_y = next(iter(cross_loader))
print("\none cross-encoder input, decoded:")
print(tokenizer.decode(demo_enc["input_ids"][0], skip_special_tokens=False)[:240])
print("label for that pair:", float(demo_y[0]))

cross = CrossEncoder(MODEL_NAME).to(DEVICE)
cross_opt = torch.optim.AdamW(cross.parameters(), lr=LR)
bce = nn.BCEWithLogitsLoss()

cross.train()
step, running = 0, []
while step < CROSS_STEPS:
    for enc, y in cross_loader:
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        y = y.to(DEVICE)
        cross_opt.zero_grad(set_to_none=True)
        loss = bce(cross(enc), y)
        loss.backward()
        cross_opt.step()
        running.append(loss.detach().item())
        step += 1
        if step % 250 == 0:
            print(f"step {step:4d}  loss {np.mean(running[-50:]):.4f}")
        if step >= CROSS_STEPS:
            break

print("\ncross-encoder trained. steps =", step)
print(f"first 50 steps avg {np.mean(running[:50]):.4f} -> "
      f"last 50 steps avg {np.mean(running[-50:]):.4f}")

## Cell 12 — Re-rank the shortlist

The towers cast a wide net; the cross-encoder sorts the catch. We only score the
`TOP_K_RETRIEVE` candidates per topic, never the whole library — that is what makes an
expensive model affordable.

We report three orderings so the comparison is honest:

- **retriever** — the cosine ordering the towers already produced,
- **cross-encoder** — the shortlist re-sorted by the pair score alone,
- **blend** — both signals added after z-scoring, which is what medal Kaggle solutions do.

The blend is worth knowing because the two models are wrong in different ways, though on
this small classroom run it usually only ties the cross-encoder rather than beating it.

In [ ]:
@torch.no_grad()
def cross_score(topic_text, candidate_ids, batch_size=64):
    cross.eval()
    scores = []
    for i in range(0, len(candidate_ids), batch_size):
        chunk = candidate_ids[i:i + batch_size]
        enc = tokenizer([topic_text] * len(chunk),
                        [content_text_by_id.get(c, "") for c in chunk],
                        padding=True, truncation=True, max_length=MAX_LEN,
                        return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        scores.append(cross(enc).float().cpu().numpy())
    return np.concatenate(scores)


def zscore(a):
    a = np.asarray(a, dtype="float32")
    return (a - a.mean()) / (a.std() + 1e-6)


reranked, blended = {}, {}
for _, row in eval_topics.iterrows():
    tid = row["id"]
    candidates = retrieved[tid]
    cs = cross_score(row["topic_text"], candidates)
    rs = retrieved_sims[tid]
    reranked[tid] = [candidates[i] for i in np.argsort(-cs)]
    blended[tid] = [candidates[i] for i in np.argsort(-(zscore(cs) + zscore(rs)))]


def precision_at(mapping, k=TOP_K_SUBMIT):
    hit = tot = 0
    for tid, got in mapping.items():
        hit += len(gold.get(tid, set()) & set(got[:k]))
        tot += k
    return hit / max(tot, 1)


print(f"precision@{TOP_K_SUBMIT}  retriever only : {precision_at(retrieved):.3f}")
print(f"precision@{TOP_K_SUBMIT}  cross-encoder  : {precision_at(reranked):.3f}")
print(f"precision@{TOP_K_SUBMIT}  blend of both  : {precision_at(blended):.3f}")

final = reranked   # submit the cross-encoder ordering
example = eval_topics.iloc[0]
print("\ntopic:", example["topic_text"][:90])
print("top 3 after re-ranking:")
for cid in final[example["id"]][:3]:
    mark = "OK " if cid in gold.get(example["id"], set()) else "   "
    print(f"  {mark}{content_text_by_id.get(cid, '')[:80]}")

## Cell 13 — Write `submission.csv`

Two columns: `topic_id`, and `content_ids` as one space-separated string. No index column.

In [ ]:
submission = pd.DataFrame({
    "topic_id": list(final.keys()),
    "content_ids": [" ".join(v[:TOP_K_SUBMIT]) for v in final.values()],
})

if not DEMO_MODE:
    submission = (sample_sub[["topic_id"]]
                  .merge(submission, on="topic_id", how="left")
                  .fillna({"content_ids": ""}))

submission.to_csv("submission.csv", index=False)

print("rows written:", len(submission))
print(submission.head().to_string(index=False))

## Done

You built the whole retrieval stack yourself:

- **Labeled pairs** gave the training signal.
- **Two towers** from one checkpoint learned to place a topic near its own content.
- **Mean pooling + L2 normalize** turned token vectors into one searchable embedding.
- **In-batch cross-entropy** used the rest of the batch as free negatives.
- **A precomputed index** made query time a single matrix multiply.
- **A cross-encoder** re-ranked the shortlist with full pairwise attention.

Next steps to try: mine hard negatives instead of random ones, share the tower weights and
compare, or swap the brute-force scan for an approximate nearest neighbour index.